# 大規模言語モデルの応用を体験する

## メディア研究とジャーナリズムのための 3 つの使い方

メディア研究を専攻する学部生のための第 2 回ハンズオンです。第 1 回では、Qwen3-8B の中間層から「感情ベクトル」を取り出し、LLM の内部表現を観察しました。

今回は視点を少し外側に移します。LLM を「研究対象」として見るだけでなく、メディア研究の作業を助ける「研究道具」として使うと、どんなことができるのでしょうか。

今日扱うのは次の 3 つです。

1. **代理ラベリング (surrogate labeling)**: YouTube 動画のタイトルと説明文から、気候変動否認論を含むかを判定する
2. **社会シミュレーション (social simulation)**: LLM エージェントを小さなソーシャルメディア社会として動かし、専門家介入や誤情報の影響を見る
3. **メカニズム分析 (mechanism analysis)**: 気候合意と気候否認の主張が、モデル内部のどのような幾何として見えるかを観察する

このノートブックは研究論文レベルの完全な実装ではありません。目的は、皆さんが「LLM を使ったメディア研究では、こういう問いの立て方ができるのか」と直感的に理解することです。コードセルは Colab では折りたたまれて表示されます。まずは文章と図を追ってください。


## ステップ 0: 環境セットアップ

第 1 回と同じく、Google Colab の無料 T4 GPU を想定します。

上部メニューから **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んでください。下のセルはライブラリを入れ、データを読み込むための準備をします。


In [ ]:
#@title 環境セットアップ (実行してください) {display-mode: "form"}
import subprocess, sys, os, gc, json, re, math, random, urllib.request
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

IN_COLAB = "google.colab" in sys.modules

# Colab already ships with NumPy, pandas, matplotlib, scikit-learn, and networkx.
# Upgrading those core scientific packages inside a running notebook can leave
# Python with mixed old/new binary modules. That is what causes errors such as:
#   numpy._core._multiarray_umath has no attribute _blas_supports_fpe
# So we install only the LLM-specific packages here.
LLM_PACKAGES = [
    "transformers>=4.51.0,<5.0.0",
    "huggingface_hub<1.0.0",
    "tokenizers>=0.21.0,<0.22.0",
    "accelerate",
    "bitsandbytes",
    "sentencepiece",
    "japanize-matplotlib",
    "openai>=1.0.0",
]

if IN_COLAB:
    print("Colab を検出しました。必要なライブラリをインストールしています...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *LLM_PACKAGES,
    ])
    print("インストール完了")

try:
    import numpy as np
    import pandas as pd
    import torch
    import matplotlib.pyplot as plt
    import networkx as nx
    from tqdm.auto import tqdm
    from IPython.display import display, Markdown
    from sklearn.metrics import confusion_matrix, accuracy_score
    from sklearn.decomposition import PCA
    from sklearn.metrics.pairwise import cosine_similarity
except AttributeError as e:
    if "_blas_supports_fpe" in str(e):
        raise RuntimeError(
            "NumPy/scikit-learn のバイナリが現在の Colab ランタイム内で混在しています。"
            "これは前のセットアップで NumPy 系パッケージを実行中に更新した後によく起きます。"
            "Colab メニューの『ランタイム』→『セッションを再起動』を押してから、"
            "このノートブックを上から再実行してください。更新後のノートブックでは"
            "NumPy/scikit-learn を上書きしないため、再起動後は解消するはずです。"
        ) from e
    raise

try:
    import japanize_matplotlib
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path("/content/llmintro_app_cache") if IN_COLAB else Path.cwd() / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_DATA_BASE = "https://raw.githubusercontent.com/CSS-Laboratory/LLMintro/main/data/application"

def load_application_csv(filename: str) -> pd.DataFrame:
    """Load teaching data locally if available, otherwise from GitHub raw URL."""
    local_candidates = [
        Path.cwd() / "data" / "application" / filename,
        Path.cwd() / filename,
        CACHE_DIR / filename,
    ]
    for path in local_candidates:
        if path.exists():
            return pd.read_csv(path)

    url = f"{GITHUB_DATA_BASE}/{filename}"
    out = CACHE_DIR / filename
    print(f"GitHub からダウンロードします: {url}")
    urllib.request.urlretrieve(url, out)
    return pd.read_csv(out)

print(f"デバイス: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU が見つかりません。Colab では T4 GPU を選んでください。")


## 実行モード

このノートブックには、時間を短くするための設定があります。

- `LIVE_MODE = True`: Qwen3-8B を実際に読み込んで動かします
- `FAST_DEMO = True`: 授業向けの短いデモ設定です。サンプル数やシミュレーション日数を少なめにします
- `USE_QWEN_FOR_SOCIAL_SIM = True`: 社会シミュレーションの「文章例」や「要約例」に Qwen3-8B を使います

大事な変更点があります。社会シミュレーションの **数値的な信念更新** は、Qwen に直接まかせません。そこを毎回 LLM に決めさせると、ネットワークが不自然に一つの値へ潰れることがあります。今日の本筋では、信念更新は透明な数式で行い、LLM は「メディア内容を読む・要約する・分類する」部分で使います。


In [ ]:
#@title 実行モードを設定 {display-mode: "form"}
LIVE_MODE = True  #@param {type:"boolean"}
FAST_DEMO = True  #@param {type:"boolean"}
USE_QWEN_FOR_SOCIAL_SIM = True  #@param {type:"boolean"}

print(f"LIVE_MODE: {LIVE_MODE}")
print(f"FAST_DEMO: {FAST_DEMO}")
print(f"USE_QWEN_FOR_SOCIAL_SIM: {USE_QWEN_FOR_SOCIAL_SIM}  # 社会シミュレーションの文章例・要約例で使用")


## Qwen3-8B を読み込む

第 1 回と同じモデルを使います。4-bit 量子化により T4 GPU でも動きます。

ここで重要なのは、今日のモデルは「正解表を持った採点者」ではなく、**次に来るトークンの確率分布を計算する機械**だという点です。代理ラベリングでも、社会シミュレーションでも、メカニズム分析でも、この性質がずっと効いてきます。


In [ ]:
#@title Qwen3-8B を 4-bit 量子化で読み込む (数分) {display-mode: "form"}
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

if LIVE_MODE:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR / "models")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        cache_dir=CACHE_DIR / "models",
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    NUM_LAYERS = len(model.model.layers)
    HIDDEN_SIZE = model.config.hidden_size
    LAYER_TO_PROBE = int(round(NUM_LAYERS * 2 / 3))

    print("モデル読み込み完了")
    print(f"Transformer 層: {NUM_LAYERS}")
    print(f"隠れ状態の次元: {HIDDEN_SIZE}")
    print(f"今日観察する層: {LAYER_TO_PROBE}")
else:
    tokenizer = None
    model = None
    NUM_LAYERS = 36
    HIDDEN_SIZE = 4096
    LAYER_TO_PROBE = 24
    print("デモモード: モデルは読み込みません。")


In [ ]:
#@title Qwen3-8B を使うための小さな関数 {display-mode: "form"}
def render_chat(user_prompt: str, system_prompt: Optional[str] = None) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def qwen_generate(
    user_prompt: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため Qwen 生成は使えません。")
    rendered = render_chat(user_prompt, system_prompt)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def token_id_for_label(label: str) -> int:
    """Find a single-token id for a short label such as 0 or 1."""
    candidates = [label, " " + label]
    for cand in candidates:
        ids = tokenizer(cand, add_special_tokens=False)["input_ids"]
        if len(ids) == 1:
            return ids[0]
    raise ValueError(f"ラベル {label!r} が 1 トークンになりませんでした: {candidates}")


def next_token_probs_for_labels(
    user_prompt: str,
    labels: Tuple[str, ...] = ("0", "1"),
    system_prompt: Optional[str] = None,
) -> Dict[str, float]:
    """Constrain a classification task to the next token and read label probabilities."""
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため logit は読めません。")
    rendered = render_chat(user_prompt, system_prompt)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
    label_ids = [token_id_for_label(label) for label in labels]
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, label_ids].float().cpu()
    probs = torch.softmax(logits, dim=0).numpy()
    return {label: float(prob) for label, prob in zip(labels, probs)}


def parse_binary_label(text: str) -> Optional[int]:
    match = re.search(r"\b([01])\b", str(text))
    return int(match.group(1)) if match else None


def parse_json_object(text: str) -> Optional[dict]:
    match = re.search(r"\{.*\}", str(text), flags=re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

print("補助関数を定義しました。")


In [ ]:
#@title 可視化スタイルを設定 {display-mode: "form"}
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import textwrap as _textwrap

# Okabe-Ito based palette: readable for many color-vision types.
OI = {
    "blue": "#0072B2",
    "sky": "#56B4E9",
    "orange": "#E69F00",
    "green": "#009E73",
    "yellow": "#F0E442",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "black": "#111111",
    "gray": "#8A8F98",
    "light_gray": "#E9ECEF",
    "paper": "#FBFBF8",
}
BELIEF_CMAP = LinearSegmentedColormap.from_list(
    "consensus_to_denial", [OI["blue"], "#F7F4EA", OI["orange"]]
)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#2B2B2B",
    "axes.labelcolor": "#2B2B2B",
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "grid.color": "#D9DDE3",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.55,
    "lines.linewidth": 2.2,
})

def clean_ax(ax, grid="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#333333")
    ax.spines["bottom"].set_color("#333333")
    if grid:
        ax.grid(True, axis=grid, zorder=0)
    else:
        ax.grid(False)
    ax.tick_params(length=0, pad=6)
    return ax

def add_note(fig, note):
    fig.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=8.5, color="#666666")

def short_label(text, width=34):
    text = str(text).replace("\n", " ").strip()
    return _textwrap.shorten(text, width=width, placeholder="...")

def direct_label(ax, x, y, label, color, dx=0.08, dy=0.0, size=9.5):
    ax.text(x + dx, y + dy, label, color=color, fontsize=size, va="center", fontweight="bold")

def panel_label(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom")

print("可視化スタイルを設定しました。")


## このノートブックの図の読み方

図は、授業中にスクリーンで見ても読みやすいように、少し編集者的なスタイルに揃えています。

- **青**: 気候科学の合意側、または否認論ではないもの
- **オレンジ**: 否認論、誤情報、またはリスクが高いもの
- **グレー**: 背景、比較対象、迷いやすい境界
- 線グラフでは、できるだけ凡例に頼らず、線の近くに直接ラベルを置きます
- 確率や信念は、基本的に `0` から `1` の同じスケールで読みます

この方針は、科学図では「見せたいメッセージを先に決める」「不要な装飾を減らす」「色だけに頼らない」「カラーマップは知覚的に読みやすいものを選ぶ」という考え方に基づいています。


---

# セクション 1: 代理ラベリング

## 直感: 1 人のアルバイトに 1 回だけ聞くのは危ない

内容分析では、人間のコーダーがコードブックを読んで、記事、投稿、動画などにラベルを付けます。LLM を使うと、この作業の一部を自動化できます。これをここでは **代理ラベリング** と呼びます。

ただし、LLM の出力は「答えそのもの」ではありません。モデルは、プロンプトを読んだあとに次のトークンの確率分布を作り、そこから 1 つの文字列を出します。

つまり、同じ動画に対して 1 回だけ `0` や `1` を出させると、確率分布の一部をたまたま見ているだけかもしれません。

研究で使うなら、少なくとも次のどちらかを考えます。

1. **複数回聞く**: 同じ対象を何度か分類させ、分布や不一致を見る
2. **logit を読む**: `0` と `1` の次トークン確率を直接比べ、モデルの迷いを見る


## データとコードブック

参照資料にある YouTube 動画サンプルと、気候変動否認論のコードブックを使います。

ラベル `denialism` は、人間コーディングによる参照ラベルです。`0` は否認論なし、`1` は否認論ありを意味します。中間値は、コーダー間の不一致や部分的な該当を表すものとして扱います。


In [ ]:
#@title YouTube サンプルとコードブックを読み込む {display-mode: "form"}
codebook_df = load_application_csv("climate_denial_codebook.csv")
sample_df = load_application_csv("climate_youtube_sample.csv")

print("コードブック:", codebook_df.shape)
print("YouTube サンプル:", sample_df.shape)
print("denialism の分布")
print(sample_df["denialism"].value_counts(dropna=False).sort_index())

display(codebook_df.head(10))
display(sample_df[["title", "channelTitle", "denialism", "conspiracy"]].head(8))


In [ ]:
#@title ラベリング用プロンプトを作る {display-mode: "form"}
def build_codebook_text(df: pd.DataFrame) -> str:
    lines = []
    for _, row in df.iterrows():
        parent = "" if pd.isna(row.get("Parent")) else f" (parent: {row['Parent']})"
        lines.append(f"- {row['Name']}{parent}: {row['Explanation']}")
    return "\n".join(lines)

CODEBOOK_TEXT = build_codebook_text(codebook_df)

LABEL_SYSTEM_PROMPT = """You are assisting a media studies content analysis project. Follow the codebook exactly. If the evidence is insufficient, choose 0."""


def build_denialism_prompt(row: pd.Series) -> str:
    title = str(row.get("title", ""))[:500]
    description = str(row.get("description", ""))[:1800]
    return f"""
以下の YouTube 動画が、気候変動否認論を含むかを判定してください。

コードブック:
{CODEBOOK_TEXT}

判定規則:
- 1 = タイトルまたは説明文に、コードブックのいずれかに該当する気候変動否認論が含まれる
- 0 = 気候変動について述べていても、否認論ではない。または情報が足りない
- 出力は 0 または 1 の 1 文字だけ

タイトル:
{title}

説明文:
{description}

ラベル:
""".strip()

print(build_denialism_prompt(sample_df.iloc[0])[:1200])


## 1 回だけ聞いたラベルを見る

まずは一番単純な方法を試します。LLM にコードブックと動画情報を渡し、`0` か `1` を 1 回だけ返してもらいます。

これは便利ですが、研究データとして使うには危険です。なぜなら、生成は確率的で、プロンプトや温度設定によって揺れることがあるからです。


In [ ]:
#@title 1 件だけ、1 回だけラベルを生成する {display-mode: "form"}
# 人間ラベルが中間値に近いものを、曖昧な例として選ぶ
ambiguous_idx = int((sample_df["denialism"].fillna(0) - 0.5).abs().sort_values().index[0])
row = sample_df.loc[ambiguous_idx]

print("対象タイトル:")
print(row["title"])
print(f"人間ラベル denialism: {row['denialism']}")
print()

if LIVE_MODE:
    raw = qwen_generate(
        build_denialism_prompt(row),
        system_prompt=LABEL_SYSTEM_PROMPT,
        max_new_tokens=8,
        temperature=0.7,
    )
    print("Qwen の出力:", repr(raw))
    print("パースしたラベル:", parse_binary_label(raw))
else:
    print("デモモード: ここでは実生成をスキップします。")


## 同じ対象を複数回聞く

同じ動画に複数回ラベルを付けさせると、モデルがどれくらい迷っているかが見えます。

たとえば 7 回中 7 回 `1` なら、少なくともこのプロンプトでは強く否認論だと見ています。7 回中 `0` と `1` が割れるなら、その対象はコードブック上も曖昧か、プロンプトが十分に明確でない可能性があります。


In [ ]:
#@title 同じ動画を複数回ラベリングする {display-mode: "form"}
N_REPEATS = 5 if FAST_DEMO else 9

if LIVE_MODE:
    repeated = []
    for i in tqdm(range(N_REPEATS), desc="同じ対象を複数回分類"):
        raw = qwen_generate(
            build_denialism_prompt(row),
            system_prompt=LABEL_SYSTEM_PROMPT,
            max_new_tokens=8,
            temperature=0.9,
        )
        repeated.append({"trial": i + 1, "raw_output": raw, "label": parse_binary_label(raw)})
    repeated_df = pd.DataFrame(repeated)
else:
    rng = np.random.default_rng(SEED)
    repeated_df = pd.DataFrame({
        "trial": range(1, N_REPEATS + 1),
        "raw_output": rng.choice(["0", "1"], size=N_REPEATS, p=[0.35, 0.65]),
    })
    repeated_df["label"] = repeated_df["raw_output"].astype(int)

display(repeated_df)
print("平均ラベル =", repeated_df["label"].dropna().mean())


## logit から確率を見る

次に、`0` と `1` を実際に生成させるのではなく、最後の位置でモデルが `0` と `1` にどのくらいの確率を置いているかを読みます。

これは「モデルの心の中を完全に読む」ことではありませんが、1 回の生成結果よりは、分類の不確実性を扱いやすくなります。


In [ ]:
#@title 0/1 の次トークン確率を読む {display-mode: "form"}
if LIVE_MODE:
    probs = next_token_probs_for_labels(
        build_denialism_prompt(row),
        labels=("0", "1"),
        system_prompt=LABEL_SYSTEM_PROMPT,
    )
else:
    probs = {"0": 0.32, "1": 0.68}

print("次トークン確率")
for label, prob in probs.items():
    print(f"  P({label}) = {prob:.3f}")
print("モデルが見ている denialism 確率の近似:", f"{probs['1']:.3f}")


## 小さなサンプル全体を分類する

授業用なので、全部ではなく一部だけ分類します。ここでは `P(1)` を使い、0.5 以上なら「否認論あり」とします。

研究で本当に使うなら、しきい値は固定でよいとは限りません。人間ラベルとの比較、コーダー間一致、検証用データでの較正が必要です。


In [ ]:
#@title 小さなサンプルを logit で分類する {display-mode: "form"}
N_POS = 4 if FAST_DEMO else 8
N_NEG = 6 if FAST_DEMO else 12

positive = sample_df[sample_df["denialism"].fillna(0) > 0].head(N_POS)
negative = sample_df[sample_df["denialism"].fillna(0) == 0].head(N_NEG)
label_subset = pd.concat([positive, negative], ignore_index=True)

records = []
if LIVE_MODE:
    for _, item in tqdm(label_subset.iterrows(), total=len(label_subset), desc="logit 分類"):
        probs = next_token_probs_for_labels(
            build_denialism_prompt(item),
            labels=("0", "1"),
            system_prompt=LABEL_SYSTEM_PROMPT,
        )
        p1 = probs["1"]
        records.append({
            "title": item["title"],
            "human": int(float(item["denialism"]) >= 0.5),
            "p_denialism": p1,
            "model": int(p1 >= 0.5),
        })
else:
    rng = np.random.default_rng(SEED)
    for _, item in label_subset.iterrows():
        human = int(float(item["denialism"]) >= 0.5)
        p1 = float(np.clip(0.25 + 0.55 * human + rng.normal(0, 0.15), 0.02, 0.98))
        records.append({"title": item["title"], "human": human, "p_denialism": p1, "model": int(p1 >= 0.5)})

label_results = pd.DataFrame(records)
display(label_results[["title", "human", "p_denialism", "model"]])

acc = accuracy_score(label_results["human"], label_results["model"])
cm = confusion_matrix(label_results["human"], label_results["model"], labels=[0, 1])
print(f"Accuracy: {acc:.2f}")
print("Confusion matrix rows=human, cols=model [0,1]:")
print(cm)


In [ ]:
#@title 図 1: 人間ラベルとモデル確率の比較 {display-mode: "form"}
plot_df = label_results.copy().sort_values("p_denialism").reset_index(drop=True)
y = np.arange(len(plot_df))
colors = [OI["orange"] if h == 1 else OI["blue"] for h in plot_df["human"]]

fig, ax = plt.subplots(figsize=(10.2, 5.6))
ax.axvspan(0.45, 0.55, color=OI["light_gray"], alpha=0.85, zorder=0)
ax.axvline(0.5, color="#222222", linestyle="--", linewidth=1.2, zorder=1)
ax.hlines(y, 0, plot_df["p_denialism"], color="#C9CED6", linewidth=3, zorder=1)
ax.scatter(plot_df["p_denialism"], y, s=95, c=colors, edgecolor="white", linewidth=1.2, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels([short_label(t, 42) for t in plot_df["title"]])
ax.set_xlim(0, 1.02)
ax.set_xlabel("Qwen が置いた P(denialism = 1)")
ax.set_title("代理ラベリングは『0/1 の答え』ではなく『不確実性つきの判定』として読む")
clean_ax(ax, grid="x")

ax.text(0.5, len(plot_df)-0.25, "判定しきい値", ha="center", va="bottom", fontsize=9, color="#333333")
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=OI["blue"], markeredgecolor="white", markersize=9, label="人間ラベル: 否認論なし"),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=OI["orange"], markeredgecolor="white", markersize=9, label="人間ラベル: 否認論あり"),
    Patch(facecolor=OI["light_gray"], edgecolor='none', label="迷いやすい境界帯"),
]
ax.legend(handles=legend_handles, loc="lower right")
add_note(fig, "注: 小さな授業用サンプル。研究では別データでしきい値を較正する。")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


## OpenAI API のスモークテスト

全員が API キーを持っているとは限らないので、ここでは実際の API 呼び出しはデフォルトで行いません。

ただし、研究現場では次のように、API にコードブック、対象テキスト、構造化出力の指定を渡してラベリングできます。OpenAI の現在の推奨は Responses API です。下のセルは `DRY RUN` として、送るリクエストの形だけを表示します。


In [ ]:
#@title OpenAI API スモークテスト (デフォルトでは送信しない) {display-mode: "form"}
RUN_OPENAI_SMOKE_TEST = False  #@param {type:"boolean"}
OPENAI_MODEL = "gpt-5.5"  #@param {type:"string"}

from pprint import pprint

smoke_row = sample_df.iloc[0]
smoke_input = build_denialism_prompt(smoke_row)

request_payload = {
    "model": OPENAI_MODEL,
    "instructions": "You are assisting a media studies content analysis project. Follow the codebook exactly.",
    "input": [
        {"role": "user", "content": smoke_input}
    ],
    "text": {
        "format": {
            "type": "json_schema",
            "name": "climate_denialism_label",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "denialism": {"type": "integer", "enum": [0, 1]},
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                    "evidence": {"type": "string"},
                },
                "required": ["denialism", "confidence", "evidence"],
                "additionalProperties": False,
            },
        }
    },
}

if RUN_OPENAI_SMOKE_TEST:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY が環境変数にありません。Colab Secrets を使ってください。")
    from openai import OpenAI
    client = OpenAI()
    response = client.responses.create(**request_payload)
    print(response.output_text)
else:
    print("DRY RUN: 実際の API 呼び出しは行っていません。送信するならこの形です。")
    pprint(request_payload, width=100)


## セクション 1 のまとめ

代理ラベリングは、メディア研究で非常に強力です。大量のニュース、投稿、動画説明文を、人間だけで読むより速く処理できます。

しかし、LLM ラベルを「人間ラベルの完全な代用品」と見なすのは危険です。

大事な実務上の考え方は次の 3 つです。

1. 1 回の返答ではなく、**確率や複数回の揺れ**を見る
2. コードブックを明確にし、人間ラベルで**較正**する
3. モデルが間違えやすい境界事例を、研究者が最後に確認する


---

# セクション 2: 社会シミュレーション

## 直感: 小さな人工社会で「もしも」を試す

社会シミュレーションでは、1 人ひとりのユーザーをエージェントとして表し、つながり、信念、記憶、役割を持たせます。

FUSE という研究枠組みは、LLM エージェントを使って「真実のニュースが、人々の相互作用を通じてどのように歪んでいくか」を調べます。あなたの Expert Strategy 実験では、20 人のネットワーク、15 日間、中心的な専門家 1 人、複数の専門家戦略 (`authoritize`, `debunk`, `educate`, `random`, `baseline`) という設計になっていました。

元になった論文は、Liu et al. (2025) **The Stepwise Deception: Simulating the Evolution from True News to Fake News with LLM Agents** です。論文では FUSE = *Fake news evolUtion Simulation framEwork* と呼ばれる枠組みを提案し、拡散者・コメント型・検証者・傍観者という役割、記憶、ネットワーク、介入戦略を組み合わせています。興味がある人は [arXiv:2410.19064](https://arxiv.org/abs/2410.19064) を読んでください。

授業用ノートブックでは、この考え方を小さく透明にします。ここで一番見せたいのは、LLM が魔法の予測機械になることではなく、**ネットワーク、信念、情報源、拒否反応を組み合わせると、メディア環境の仮説を実験できる**という点です。

今回は数値的な信念更新を、あえて見える数式で行います。Qwen は後で「投稿や要約を読む」部分に使います。


In [ ]:
#@title 小さな FUSE 風シミュレーションの部品 {display-mode: "form"}
@dataclass
class MiniAgent:
    id: int
    role: str
    belief: float  # 0 = climate consensus, 1 = strong climate denialism
    receptiveness: float
    message: str = ""

ROLE_EFFECT = {
    "spreader": 1.15,
    "commentator": 1.00,
    "verifier": 0.70,
    "bystander": 0.50,
}

ROLE_JP = {
    "spreader": "拡散者",
    "commentator": "コメント型",
    "verifier": "検証者",
    "bystander": "傍観者",
}

CONSENSUS_MESSAGE = "Human activities are the main driver of recent climate change."
DENIAL_MESSAGE = "Climate change is mostly natural variability and policy costs are exaggerated."

def edge_key(u: int, v: int) -> Tuple[int, int]:
    return tuple(sorted((int(u), int(v))))

def generate_polarized_belief_scores(
    n_users: int,
    polarization_ratio: float = 0.86,
    ub: float = 1.0,
    lb: float = 0.0,
    seed: int = SEED,
) -> np.ndarray:
    """Gaussian-mixture belief scores, adapted from the Expert Strategy notebook."""
    rng = np.random.default_rng(seed)
    n_polarized = int(round(n_users * polarization_ratio))
    n_polarized = min(max(2, n_polarized), n_users)
    n_centrist = n_users - n_polarized
    n_low = n_polarized // 2
    n_high = n_polarized - n_low
    center = (ub + lb) / 2

    # Same idea as the previous notebook: two polarized modes plus a smaller centrist group.
    # The locations are slightly softened so the classroom figure is readable rather than all 0/1.
    scale = ub - lb
    scores_c = rng.normal(loc=center, scale=0.08 * scale, size=n_centrist)
    scores_l = rng.normal(loc=lb + 0.12 * scale, scale=0.08 * scale, size=n_low)
    scores_u = rng.normal(loc=ub - 0.12 * scale, scale=0.08 * scale, size=n_high)
    scores = np.clip(np.concatenate([scores_c, scores_l, scores_u]), lb, ub)
    rng.shuffle(scores)
    return scores

def create_mini_agents(n: int, seed: int = SEED, beliefs: Optional[np.ndarray] = None) -> List[MiniAgent]:
    """Create agents whose initial beliefs come from the polarized mixture."""
    rng = np.random.default_rng(seed + 1)
    if beliefs is None:
        beliefs = generate_polarized_belief_scores(n, seed=seed)
    roles = rng.choice(list(ROLE_EFFECT), size=n, p=[0.30, 0.45, 0.15, 0.10])

    agents = []
    for i, (belief, role) in enumerate(zip(beliefs, roles)):
        receptiveness = float(np.clip(0.82 - 0.32 * abs(float(belief) - 0.5) + rng.normal(0, 0.05), 0.25, 0.95))
        message = CONSENSUS_MESSAGE if belief < 0.5 else DENIAL_MESSAGE
        agents.append(MiniAgent(i, str(role), float(belief), receptiveness, message))
    return agents

def ensure_connectivity_with_bridges(G: nx.Graph, scores: np.ndarray) -> nx.Graph:
    """Bridge disconnected components, following the spirit of the previous notebook."""
    G = G.copy()
    bridge_edges = {edge_key(*e) for e in G.graph.get("bridge_edges", [])}
    if nx.number_connected_components(G) <= 1:
        G.graph["bridge_edges"] = sorted(bridge_edges)
        return G

    components = sorted(nx.connected_components(G), key=len, reverse=True)
    main_component = set(components[0])
    for island in components[1:]:
        # Choose the least distant cross-component pair so the bridge is plausible.
        pairs = [(abs(scores[u] - scores[v]), int(u), int(v)) for u in island for v in main_component]
        _, u, v = min(pairs, key=lambda item: item[0])
        G.add_edge(u, v)
        bridge_edges.add(edge_key(u, v))
        main_component.update(island)
    G.graph["bridge_edges"] = sorted(bridge_edges)
    return G

def ensure_cross_community_bridges(
    G: nx.Graph,
    scores: np.ndarray,
    min_bridges: int = 2,
    low_cut: float = 0.42,
    high_cut: float = 0.58,
) -> nx.Graph:
    """Make the two belief communities clustered but not completely isolated."""
    G = G.copy()
    bridge_edges = {edge_key(*e) for e in G.graph.get("bridge_edges", [])}
    lows = [i for i, s in enumerate(scores) if s <= low_cut]
    highs = [i for i, s in enumerate(scores) if s >= high_cut]
    if not lows or not highs:
        G.graph["bridge_edges"] = sorted(bridge_edges)
        return G

    existing = [edge_key(u, v) for u, v in G.edges() if (u in lows and v in highs) or (u in highs and v in lows)]
    bridge_edges.update(existing)
    candidates = sorted(
        [(abs(scores[u] - scores[v]), int(u), int(v)) for u in lows for v in highs if not G.has_edge(u, v)],
        key=lambda item: item[0],
    )
    idx = 0
    while len([e for e in bridge_edges if e[0] in lows and e[1] in highs or e[1] in lows and e[0] in highs]) < min_bridges and idx < len(candidates):
        _, u, v = candidates[idx]
        G.add_edge(u, v)
        bridge_edges.add(edge_key(u, v))
        idx += 1
    G.graph["bridge_edges"] = sorted(bridge_edges)
    return G

def build_homophily_graph(
    agents: List[MiniAgent],
    target_degree: int = 3,
    homophily_strength: float = 6.0,
    seed: int = SEED,
    min_cross_bridges: int = 2,
    homophily: Optional[float] = None,
) -> nx.Graph:
    """Homophily network: similar beliefs connect often, with a few explicit bridge ties."""
    # Backward-compatible alias used by earlier versions of this notebook.
    if homophily is not None:
        homophily_strength = homophily
    rng = np.random.default_rng(seed)
    scores = np.array([a.belief for a in agents])
    distances = np.abs(scores[:, None] - scores[None, :])
    raw_probabilities = np.exp(-homophily_strength * (distances ** 2))
    np.fill_diagonal(raw_probabilities, 0)

    desired_weight_sum = len(agents) * target_degree
    scaling_factor = desired_weight_sum / (raw_probabilities.sum() + 1e-9)
    final_probs = np.clip(raw_probabilities * scaling_factor, 0, 1)

    random_draws = rng.random(size=final_probs.shape)
    adjacency = np.triu((random_draws < final_probs).astype(int), 1)
    G = nx.from_numpy_array(adjacency + adjacency.T)
    for i, score in enumerate(scores):
        G.nodes[i]["score"] = float(score)

    G.graph["bridge_edges"] = []
    G = ensure_connectivity_with_bridges(G, scores)
    G = ensure_cross_community_bridges(G, scores, min_bridges=min_cross_bridges)
    return G

def plot_belief_network(G: nx.Graph, agents: List[MiniAgent], title: str, ax=None, pos=None, expert_node: Optional[int] = None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.8, 4.8))
    if pos is None:
        pos = nx.spring_layout(G, seed=SEED, k=0.55)
    beliefs = np.array([a.belief for a in agents])
    bridge_edges = [tuple(e) for e in G.graph.get("bridge_edges", []) if G.has_edge(*e)]
    bridge_keys = {edge_key(*e) for e in bridge_edges}
    regular_edges = [e for e in G.edges() if edge_key(*e) not in bridge_keys]
    node_sizes = [330 if a.role == "verifier" else 260 for a in agents]

    nx.draw_networkx_edges(G, pos, edgelist=regular_edges, ax=ax, alpha=0.18, edge_color="#7A8088", width=1.0)
    if bridge_edges:
        nx.draw_networkx_edges(G, pos, edgelist=bridge_edges, ax=ax, alpha=0.85, edge_color=OI["green"], width=2.0, style="--")
    nodes = nx.draw_networkx_nodes(
        G, pos, ax=ax, node_color=beliefs, cmap=BELIEF_CMAP, vmin=0, vmax=1,
        node_size=node_sizes, edgecolors="white", linewidths=1.2,
    )
    nx.draw_networkx_labels(G, pos, labels={a.id: str(a.id) for a in agents}, font_size=7.8, font_color="#202020", ax=ax)
    if expert_node is not None:
        ax.scatter([pos[expert_node][0]], [pos[expert_node][1]], marker="*", s=360, c=OI["green"], edgecolors="white", linewidths=1.4, zorder=5)
    ax.set_title(title)
    ax.axis("off")
    return nodes

agents_demo = create_mini_agents(20 if not FAST_DEMO else 16)
G_demo = build_homophily_graph(agents_demo, min_cross_bridges=2)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), gridspec_kw={"width_ratios": [1, 1.15]})
axes[0].hist([a.belief for a in agents_demo], bins=np.linspace(0, 1, 9), color=OI["gray"], edgecolor="white")
axes[0].axvline(0.66, color=OI["orange"], linestyle="--", linewidth=1.5)
axes[0].set_title("初期信念: 2 つの山と少数の中間層")
axes[0].set_xlabel("denialism belief")
axes[0].set_ylabel("agents")
clean_ax(axes[0])
nodes = plot_belief_network(G_demo, agents_demo, "初期ネットワーク: クラスタ間に橋渡しの tie がある", ax=axes[1])
axes[1].plot([], [], color=OI["green"], linestyle="--", linewidth=2, label="bridge tie")
axes[1].legend(loc="lower left")
cbar = fig.colorbar(nodes, ax=axes[1], fraction=0.046, pad=0.02)
cbar.set_label("0 = 科学的合意, 1 = 強い否認論")
add_note(fig, "緑の破線は橋渡し tie。前回の Expert Strategy notebook と同じく、信念の近さで確率的に結び、孤立成分を橋でつなぐ。")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


## 問い 1: 専門家はネットワークを分極化させることがあるか

ここでは、気候科学の専門家が毎日「人為的気候変動についての正確な情報」を出す状況を考えます。

直感的には、専門家が正しい情報を出せば、全体の否認論は弱まるはずです。しかし、ソーシャルメディアでは次の 3 つが同時に起きます。

1. **社会的影響**: 人は近くの人の投稿に少しずつ寄る
2. **同質性**: 似た信念の人とつながりやすく、違いが大きい相手とは離れやすい
3. **拒否反応**: 専門家の情報が自分の信念から遠すぎると、情報を受け取らず、逆に強い立場へ動くことがある

この設定では、専門家介入は平均的な否認論を少し下げる一方で、分極化を強めることがあります。中間層が専門家に近づき、強い否認論層が拒否するためです。


In [ ]:
#@title 専門家介入つき信念更新 {display-mode: "form"}
EXPERT_TRUTH_POSITION = 0.0
EXPERT_REJECTION_THRESHOLD = 0.66

def bounded_confidence_step(agent: MiniAgent, neighbor_beliefs: List[float]) -> float:
    """Social influence: nearby beliefs matter more than distant beliefs."""
    if len(neighbor_beliefs) == 0:
        return 0.0
    vals = np.array(neighbor_beliefs)
    diffs = np.abs(vals - agent.belief)
    weights = np.exp(-(diffs ** 2) / (2 * 0.23 ** 2))
    social_target = float(np.sum(vals * weights) / (np.sum(weights) + 1e-9))
    return 0.20 * agent.receptiveness * ROLE_EFFECT.get(agent.role, 1.0) * (social_target - agent.belief)

def expert_influence_step(agent: MiniAgent) -> Tuple[float, bool]:
    """Expert broadcast: accepted by moderate users, rejected by far-away users."""
    if agent.belief < EXPERT_REJECTION_THRESHOLD:
        return 0.16 * agent.receptiveness * (EXPERT_TRUTH_POSITION - agent.belief), False
    return 0.055 * (1.0 - agent.belief), True

def update_message(agent: MiniAgent):
    if agent.belief > 0.72:
        agent.message = "Climate science is uncertain, and policy costs are probably exaggerated."
    elif agent.belief < 0.32:
        agent.message = "Human-caused climate change is well supported and policy should follow evidence."
    else:
        agent.message = "Climate change is serious, but policy costs and evidence should both be discussed."

def maybe_rewire(G: nx.Graph, agents: List[MiniAgent], rng: np.random.Generator) -> nx.Graph:
    G = G.copy()
    # Cut ties when belief distance becomes too large.
    to_remove = []
    for u, v in list(G.edges()):
        if abs(agents[u].belief - agents[v].belief) > 0.46 and rng.random() < 0.35:
            to_remove.append((u, v))
    G.remove_edges_from(to_remove)

    # Add ties through friends-of-friends when beliefs are close.
    for u in list(G.nodes()):
        if G.degree(u) >= 4:
            continue
        candidates = set()
        for nbr in G.neighbors(u):
            candidates.update(G.neighbors(nbr))
        candidates.discard(u)
        candidates = [v for v in candidates if not G.has_edge(u, v) and G.degree(v) < 5]
        rng.shuffle(candidates)
        for v in candidates[:3]:
            diff = abs(agents[u].belief - agents[v].belief)
            if rng.random() < 0.16 * np.exp(-8 * diff * diff):
                G.add_edge(u, v)
                break

    # Keep the visualization connected enough to read.
    for i in list(G.nodes()):
        if G.degree(i) == 0:
            nearest = int(np.argsort([999 if i == k else abs(agents[i].belief - agents[k].belief) for k in range(len(agents))])[0])
            G.add_edge(i, nearest)
    return G

print("透明な信念更新ルールを定義しました。")


In [ ]:
#@title 動的ネットワークと専門家拒否を動かす {display-mode: "form"}
def run_expert_simulation(with_expert: bool, seed: int = SEED) -> Tuple[pd.DataFrame, nx.Graph, List[MiniAgent], Dict[int, List[float]]]:
    rng = np.random.default_rng(seed)
    n_agents = 20 if not FAST_DEMO else 16
    n_days = 10 if not FAST_DEMO else 8
    agents = create_mini_agents(n_agents, seed=seed)
    G = build_homophily_graph(agents, target_degree=3, seed=seed)
    snapshots = {0: [a.belief for a in agents]}
    history = []

    for day in range(n_days + 1):
        beliefs = np.array([a.belief for a in agents])
        edge_diffs = [abs(agents[u].belief - agents[v].belief) for u, v in G.edges()]
        history.append({
            "day": day,
            "condition": "expert broadcast" if with_expert else "peer network only",
            "mean_denialism": float(beliefs.mean()),
            "polarization_var": float(beliefs.var()),
            "extreme_share": float((beliefs > 0.75).mean()),
            "edge_disagreement": float(np.mean(edge_diffs)) if edge_diffs else 0.0,
            "expert_refusal_rate": float((beliefs >= EXPERT_REJECTION_THRESHOLD).mean()) if with_expert else 0.0,
            "edge_count": int(G.number_of_edges()),
        })
        if day == n_days:
            break

        proposed = []
        refused_flags = []
        for agent in agents:
            nbrs = list(G.neighbors(agent.id))
            neighbor_beliefs = [agents[j].belief for j in nbrs]
            dx = bounded_confidence_step(agent, neighbor_beliefs)
            refused = False
            if with_expert:
                expert_dx, refused = expert_influence_step(agent)
                dx += expert_dx
            dx += rng.normal(0, 0.012)
            proposed.append(float(np.clip(agent.belief + dx, 0, 1)))
            refused_flags.append(refused)

        for agent, belief in zip(agents, proposed):
            agent.belief = belief
            update_message(agent)

        G = maybe_rewire(G, agents, rng)
        if day + 1 in {3, n_days}:
            snapshots[day + 1] = [a.belief for a in agents]

    return pd.DataFrame(history), G, agents, snapshots

baseline_hist, baseline_G, baseline_agents, baseline_snapshots = run_expert_simulation(False, seed=SEED)
expert_hist, expert_G, expert_agents, expert_snapshots = run_expert_simulation(True, seed=SEED)
expert_results = pd.concat([baseline_hist, expert_hist], ignore_index=True)

display(expert_results.round(3))


In [ ]:
#@title 図 2: 専門家介入と分極化 {display-mode: "form"}
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2))
axes = axes.ravel()
colors = {"peer network only": OI["blue"], "expert broadcast": OI["orange"]}
labels_jp = {"peer network only": "専門家なし", "expert broadcast": "専門家あり"}

for condition, group in expert_results.groupby("condition"):
    color = colors[condition]
    label = labels_jp[condition]
    axes[0].plot(group["day"], group["mean_denialism"], marker="o", color=color, label=label)
    axes[1].plot(group["day"], group["polarization_var"], marker="o", color=color, label=label)
    axes[2].plot(group["day"], group["expert_refusal_rate"], marker="o", color=color, label=label)

axes[0].set_title("平均: 専門家は全体の否認論を少し下げる")
axes[0].set_ylabel("平均 denialism belief")
axes[1].set_title("分散: しかし分極化は強まることがある")
axes[1].set_ylabel("belief variance")
axes[2].set_title("拒否: 強い否認論層は専門家情報を受け取りにくい")
axes[2].set_ylabel("refusal rate")
for ax in axes[:3]:
    ax.set_xlabel("day")
    clean_ax(ax)
axes[0].legend(loc="best")

# Snapshot distribution: before vs after, expert condition.
initial = expert_snapshots[min(expert_snapshots.keys())]
final_day = max(expert_snapshots.keys())
final = expert_snapshots[final_day]
bins = np.linspace(0, 1, 9)
axes[3].hist(initial, bins=bins, color="#B8BDC7", edgecolor="white", alpha=0.90, label="day 0")
axes[3].hist(final, bins=bins, color=OI["orange"], edgecolor="white", alpha=0.72, label=f"day {final_day}")
axes[3].axvline(EXPERT_REJECTION_THRESHOLD, color="#333333", linestyle="--", linewidth=1.2)
axes[3].text(EXPERT_REJECTION_THRESHOLD + 0.02, axes[3].get_ylim()[1] * 0.88, "拒否しやすい領域", fontsize=9, color="#333333")
axes[3].set_title("専門家あり条件: 中間層と拒否層が分かれる")
axes[3].set_xlabel("denialism belief")
axes[3].set_ylabel("agents")
axes[3].legend()
clean_ax(axes[3])

add_note(fig, "授業用モデル: 信念更新は透明な数式。Qwen は数値更新ではなく、内容の分類・要約で使う。")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

# Network snapshots, with a shared layout for comparison.
pos = nx.spring_layout(expert_G, seed=SEED, k=0.55)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8))
nodes_a = plot_belief_network(baseline_G, baseline_agents, "最終ネットワーク: 専門家なし", ax=axes[0], pos=pos)
nodes_b = plot_belief_network(expert_G, expert_agents, "最終ネットワーク: 専門家あり", ax=axes[1], pos=pos)
cbar = fig.colorbar(nodes_b, ax=axes, fraction=0.035, pad=0.02)
cbar.set_label("0 = 科学的合意, 1 = 強い否認論")
add_note(fig, "色の近いノード同士が残りやすい。専門家介入後、片側が合意へ近づき、片側が拒否領域に残る。")
plt.show()


## 結果の読み方

このシミュレーションは、現実社会を予測するものではありません。むしろ、問いを組み立てるための装置です。

注目してほしいのは、専門家情報が常に単純な「良い効果」を持つとは限らない、という構造です。

- 中程度の信念を持つユーザーには、専門家情報が効きやすい
- 強い否認論を持つユーザーには、専門家情報が遠すぎて拒否されることがある
- ネットワークが同質な相手へ再接続すると、分極化が維持されやすい

メディア研究では、ここから「どんなメッセージ設計なら反発が弱まるか」「専門家は中心にいるべきか、周辺に複数いるべきか」といった研究設計に進めます。


## 問い 2: AI アシスタントは誤情報に対して脆弱か

次に、各ユーザーが AI アシスタントに「今日のフィードを要約して」と頼む状況を考えます。

もし悪意あるユーザーが毎日、気候否認論を投稿し続けたら、AI アシスタントの要約を通じて、ネットワーク全体の認識はどれくらい動くでしょうか。

ここでは、アシスタントが見た投稿を短く要約し、その要約がどの程度「否認論的」に見えるかを測ります。


In [ ]:
#@title AI アシスタント要約ネットワーク {display-mode: "form"}
EVIDENCE_MESSAGES = [
    "Human activities are the main driver of recent warming according to multiple scientific assessments.",
    "Climate risks include heat waves, sea-level rise, and damage to ecosystems and public health.",
    "Policy debates can discuss costs, but the physical evidence for warming is strong.",
]

MALICIOUS_MESSAGE = (
    "Climate change is mostly natural cycles, climate models cannot be trusted, "
    "and climate policy is just an expensive political agenda."
)


def summarize_feed_with_qwen(feed_messages: List[str]) -> str:
    feed = "\n".join([f"- {m}" for m in feed_messages])
    prompt = f"""
You are an AI assistant summarizing a user's social media feed.
Summarize the feed in one neutral English sentence. Do not add facts that are not in the feed.

Feed:
{feed}

One-sentence summary:
""".strip()
    return qwen_generate(prompt, max_new_tokens=60, temperature=0.35)


def denial_probability_for_summary(summary: str) -> float:
    prompt = f"""
Does the following summary contain climate denialism?
Return 1 if it denies climate change, denies human causation, claims science is unreliable, or claims climate policy cannot work.
Return 0 otherwise. Output only 0 or 1.

Summary:
{summary}

Label:
""".strip()
    if LIVE_MODE:
        return next_token_probs_for_labels(prompt, labels=("0", "1"), system_prompt=LABEL_SYSTEM_PROMPT)["1"]
    keywords = ["natural", "models cannot", "agenda", "exaggerated", "not trusted"]
    return 0.75 if any(k in summary.lower() for k in keywords) else 0.20


def run_assistant_vulnerability(malicious: bool, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n_agents = 6 if FAST_DEMO else 10
    n_days = 3 if FAST_DEMO else 5
    agents = create_mini_agents(n_agents, seed=seed + 7)
    for a in agents:
        a.belief = float(np.clip(rng.normal(0.28, 0.08), 0, 1))
        a.message = rng.choice(EVIDENCE_MESSAGES)
    G = build_homophily_graph(agents, target_degree=3, homophily_strength=3.5, seed=seed + 7)

    rows = []
    for day in range(n_days + 1):
        rows.append({
            "day": day,
            "condition": "malicious_user" if malicious else "no_malicious_user",
            "mean_denialism": float(np.mean([a.belief for a in agents])),
            "max_denialism": float(np.max([a.belief for a in agents])),
        })
        if day == n_days:
            break

        new_states = []
        for agent in agents:
            nbrs = list(G.neighbors(agent.id))
            feed = [agents[j].message for j in nbrs]
            feed += list(rng.choice(EVIDENCE_MESSAGES, size=2, replace=True))
            if malicious:
                # 悪意あるユーザー 0 が毎日同じ方向の情報を混ぜる
                if agent.id != 0:
                    feed.append(MALICIOUS_MESSAGE)
                else:
                    feed = [MALICIOUS_MESSAGE, MALICIOUS_MESSAGE]

            if LIVE_MODE and USE_QWEN_FOR_SOCIAL_SIM:
                try:
                    summary = summarize_feed_with_qwen(feed)
                    p_denial = denial_probability_for_summary(summary)
                except Exception as e:
                    print(f"assistant summary failed, fallback used: {e}")
                    summary = " ".join(feed[:2])[:240]
                    p_denial = denial_probability_for_summary(summary)
            else:
                summary = " ".join(feed[:2])[:240]
                p_denial = denial_probability_for_summary(summary)

            updated = float(np.clip(0.70 * agent.belief + 0.30 * p_denial, 0, 1))
            new_states.append((updated, summary))

        for agent, (belief, message) in zip(agents, new_states):
            agent.belief = belief
            agent.message = message

    return pd.DataFrame(rows)

assist_clean = run_assistant_vulnerability(False, seed=SEED)
assist_attack = run_assistant_vulnerability(True, seed=SEED)
assistant_results = pd.concat([assist_clean, assist_attack], ignore_index=True)
display(assistant_results)


In [ ]:
#@title 図 3: 悪意あるユーザーの有無で比較する {display-mode: "form"}
fig, ax = plt.subplots(figsize=(8.4, 4.8))
colors = {"malicious_user": OI["orange"], "no_malicious_user": OI["blue"]}
labels = {"malicious_user": "悪意あるユーザーあり", "no_malicious_user": "悪意あるユーザーなし"}
for condition, group in assistant_results.groupby("condition"):
    ax.plot(group["day"], group["mean_denialism"], marker="o", color=colors[condition], label=labels[condition])
    direct_label(ax, group["day"].iloc[-1], group["mean_denialism"].iloc[-1], labels[condition], colors[condition], dx=0.05)
ax.set_title("AI アシスタント要約は、反復される誤情報を増幅する入口になりうる")
ax.set_xlabel("day")
ax.set_ylabel("平均 denialism score")
ax.set_ylim(0, 1)
ax.set_xlim(assistant_results["day"].min(), assistant_results["day"].max() + 1.35)
clean_ax(ax)
ax.legend().remove()
add_note(fig, "同じネットワークで比較。違いは、悪意あるユーザーが毎日フィードに否認論的投稿を混ぜるかどうか。")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

pivot = assistant_results.pivot(index="day", columns="condition", values="mean_denialism")
if {"malicious_user", "no_malicious_user"}.issubset(set(pivot.columns)):
    pivot["vulnerability_gap"] = pivot["malicious_user"] - pivot["no_malicious_user"]
    display(pivot.round(3))


## セクション 2 のまとめ

LLM エージェントの社会シミュレーションは、現実をそのまま予測する道具ではありません。

むしろ、次のような「メカニズムの仮説」を試すためのスケッチです。

- 似た信念の人とつながりやすいネットワークでは、分極化が維持されやすい
- 専門家情報は、受け手の信念と距離が大きすぎると反発を生む可能性がある
- AI アシスタントが要約を担う環境では、悪意ある投稿の繰り返しが要約の中に入り込む可能性がある

研究として進めるには、実データによる較正、複数シード、複数モデル、人間評価が必要です。


---

# セクション 3: メカニズム分析

## 直感: LLM はどこで「これは否認論だ」と判断しているのか

代理ラベリングでは、Qwen に YouTube 動画を `0/1` で判定させました。ここでは一歩進んで、判定の背後にある構造を観察します。

メカニズム分析で見たいのは、次の 2 つです。

1. **行動空間**: 出力確率として、Qwen はどの主張を否認論に近いと見ているか
2. **表現空間**: モデル内部の活性化に、合意主張と否認主張を分ける方向があるか

注意してください。これは「モデルの本当の心を読む」ことではありません。小さな実験を通じて、LLM の判断がどのような境界を持つのかを観察する方法です。


In [ ]:
#@title 気候主張の小さなバンクを作る {display-mode: "form"}
climate_claims = [
    # Consensus / non-denial
    (0.00, "consensus", 0, "Human activities, especially greenhouse gas emissions, are the primary cause of recent global warming."),
    (0.04, "consensus", 0, "Multiple independent measurements show that the planet is warming and that human emissions are the main driver."),
    (0.08, "consensus", 0, "Climate change is already increasing risks from heat waves, sea-level rise, and damage to ecosystems."),
    (0.12, "consensus", 0, "Climate policy should be debated carefully, but the scientific evidence for human-caused warming is strong."),
    (0.18, "policy concern", 0, "Some climate policies may create unfair costs for households, so governments should design support carefully."),
    (0.24, "policy concern", 0, "People can accept climate science while disagreeing about which policy is fairest or most affordable."),
    (0.30, "uncertainty", 0, "Climate models have uncertainty, and public communication should explain that uncertainty clearly."),
    (0.36, "uncertainty", 0, "There are uncertainties in regional projections, but uncertainty does not mean the overall science is false."),
    # Boundary / science trust
    (0.48, "science skepticism", 0, "Scientists disagree about some details, so journalists should avoid overstating a single study."),
    (0.55, "science skepticism", 1, "Scientists disagree so much that climate predictions should not be trusted for policy."),
    (0.62, "science skepticism", 1, "Climate researchers exaggerate warming because funding and politics reward alarmist results."),
    # Denial categories from the codebook
    (0.70, "not happening", 1, "Global temperature and sea-level trends are too small to show that climate change is a serious problem."),
    (0.74, "not us", 1, "Recent warming is mainly caused by natural variability rather than human emissions."),
    (0.78, "not us", 1, "Changes in the sun and cosmic rays explain global warming better than carbon dioxide."),
    (0.82, "not bad", 1, "Carbon dioxide is beneficial for plants and should not be treated as pollution."),
    (0.86, "solutions won't work", 1, "Climate policy will damage the economy while doing almost nothing to reduce warming."),
    (0.90, "solutions won't work", 1, "Clean energy technology cannot deliver the emissions reductions that climate activists promise."),
    (0.94, "science unreliable", 1, "Climate models are too unreliable to justify major changes to energy policy."),
    (0.98, "science unreliable", 1, "Climate change is an exaggerated political agenda promoted by biased scientists and activists."),
]
claims_df = pd.DataFrame(climate_claims, columns=["stance", "group", "denial_label", "claim"])
display(claims_df)


In [ ]:
#@title 主張の活性化を取り出す {display-mode: "form"}
def extract_pooled_activations(texts: List[str], layer_idx: int, batch_size: int = 4, pool_skip_first: int = 4) -> np.ndarray:
    if model is None:
        raise RuntimeError("LIVE_MODE=False のため活性化抽出は使えません。")
    tokenizer.padding_side = "right"
    pooled = []
    for start in tqdm(range(0, len(texts), batch_size), desc="活性化を抽出"):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        hidden = outputs.hidden_states[layer_idx + 1].float().cpu()
        mask = inputs["attention_mask"].cpu()
        for row in range(hidden.shape[0]):
            seq_len = int(mask[row].sum().item())
            begin = min(pool_skip_first, seq_len - 1)
            pooled.append(hidden[row, begin:seq_len].mean(dim=0).numpy())
    tokenizer.padding_side = "left"
    return np.stack(pooled)

claim_texts = ["Climate claim: " + c for c in claims_df["claim"].tolist()]

if LIVE_MODE:
    claim_matrix = extract_pooled_activations(claim_texts, LAYER_TO_PROBE, batch_size=4)
else:
    rng = np.random.default_rng(SEED)
    x = claims_df["stance"].to_numpy()
    claim_matrix = np.stack([x, x**2, np.sin(x * np.pi)], axis=1) + rng.normal(0, 0.04, size=(len(x), 3))

print("claim_matrix shape:", claim_matrix.shape)


In [ ]:
#@title 各主張に対する否認論確率を読む {display-mode: "form"}
def build_claim_denial_prompt(claim: str) -> str:
    return f"""
Does the following climate-related claim contain climate denialism according to the codebook?
Return 1 if it denies climate change, denies human causation, claims climate science is unreliable, or claims climate solutions cannot work.
Return 0 otherwise. Output only 0 or 1.

Claim:
{claim}

Label:
""".strip()

p_denials = []
if LIVE_MODE:
    for claim in tqdm(claims_df["claim"], desc="否認論確率"):
        p_denials.append(next_token_probs_for_labels(build_claim_denial_prompt(claim), labels=("0", "1"), system_prompt=LABEL_SYSTEM_PROMPT)["1"])
else:
    rng = np.random.default_rng(SEED)
    p_denials = np.clip(1 / (1 + np.exp(-16 * (claims_df["stance"].to_numpy() - 0.56))) + rng.normal(0, 0.03, size=len(claims_df)), 0, 1).tolist()

claims_df["qwen_p_denialism"] = p_denials
display(claims_df[["group", "stance", "denial_label", "qwen_p_denialism", "claim"]].round(3))


In [ ]:
#@title 図 4: 行動空間: Qwen はどこで境界を引くか {display-mode: "form"}
ladder_df = claims_df.sort_values("stance").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10.4, 5.2))
ax.plot(ladder_df["stance"], ladder_df["qwen_p_denialism"], color=OI["black"], linewidth=2.2, zorder=2)
colors = [OI["orange"] if y == 1 else OI["blue"] for y in ladder_df["denial_label"]]
ax.scatter(ladder_df["stance"], ladder_df["qwen_p_denialism"], c=colors, s=95, edgecolor="white", linewidth=1.2, zorder=3)
ax.axhline(0.5, color="#333333", linestyle="--", linewidth=1)
ax.axvspan(0.44, 0.64, color=OI["light_gray"], alpha=0.70, zorder=0)
ax.text(0.54, 0.08, "境界が現れやすい領域\n(science trust)", ha="center", va="bottom", fontsize=9.5, color="#555555")
ax.set_ylim(-0.04, 1.04)
ax.set_xlim(-0.03, 1.03)
ax.set_title("行動空間: 主張を少しずつ否認論へ近づけると、出力確率が急に変わる")
ax.set_xlabel("授業用 stance ladder: 0 = 科学的合意, 1 = 強い否認論")
ax.set_ylabel("Qwen P(denialism = 1)")
clean_ax(ax)
direct_label(ax, ladder_df["stance"].iloc[-1], ladder_df["qwen_p_denialism"].iloc[-1], "否認論", OI["orange"], dx=-0.17, dy=0.04)
direct_label(ax, ladder_df["stance"].iloc[0], ladder_df["qwen_p_denialism"].iloc[0], "合意", OI["blue"], dx=0.03, dy=0.05)
add_note(fig, "これは生成文ではなく、0/1 ラベルの次トークン確率。1 回の答えより、境界の位置が見えやすい。")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## 図 4 をどう読むか: stance ladder の直感と計算

図 4 は、気候主張を「合意」から「強い否認論」へ少しずつ動かしたとき、Qwen3-8B の分類確率がどのように変わるかを見ています。

直感的には、これは階段のような実験です。下の段には「人間活動が温暖化の主因である」という合意文があります。途中には「政策コストは公平に考えるべき」「モデルには不確実性がある」という、否認論ではないが境界に近い文があります。上の段には「科学者は信用できない」「自然変動が主因だ」という否認論があります。

計算では、各主張 `claim_i` について、コードブックに基づく 0/1 分類プロンプトを作ります。そしてモデルに自由に長い文章を生成させるのではなく、次の 1 トークンが `0` か `1` になる確率だけを読みます。

\[
p_i = P(y_i = 1 \mid \text{prompt}(claim_i))
= \frac{\exp z_i(1)}{\exp z_i(0) + \exp z_i(1)}
\]

ここで `1` は「否認論あり」、`0` は「否認論なし」です。`z_i(1)` と `z_i(0)` は、Qwen が最後の位置で `1` と `0` に置いた logit です。

この図が面白いのは、確率がなめらかな直線ではなく、**science trust の境界付近で急に変わる**ところです。たとえば、「科学には細部の不一致があるので、単一研究を誇張すべきではない」は低い確率に留まります。一方で、「科学者の不一致が大きすぎるので、政策に使うべきではない」は急に高い確率になります。

つまり Qwen は、単に「uncertainty」という単語に反応しているだけではありません。少なくともこの小さな例では、**不確実性を説明する言い方**と、**科学的信頼そのものを崩す言い方**をかなり鋭く分けています。

ただし、これはまだ研究結果ではありません。小さな授業用 claim bank の観察です。研究として使うには、主張数を増やし、日本語文でも試し、複数モデル・複数プロンプト・複数レイヤーで同じ境界が出るかを確認する必要があります。


## Manifold steering の発想を、小さく試す

ここからは、出力確率だけでなく、モデル内部の活性化を見ます。

元になった論文は、Wurgaft et al. (2026) **Manifold Steering Reveals the Shared Geometry of Neural Network Representation and Behavior** です。論文の主張は、ニューラルネットワークの内部表現は単なる点の集まりではなく、曲がった幾何構造を持ち、その幾何に沿って介入すると自然な行動変化が起きやすい、というものです。興味がある人は [arXiv:2605.05115](https://arxiv.org/abs/2605.05115) を読んでください。

まず、気候合意に近い主張の平均活性化と、否認論に近い主張の平均活性化を比べます。この差分を、授業用に **denial axis** と呼びます。

これは第 1 回の感情ベクトルと同じ考え方です。

- 合意主張の平均位置
- 否認主張の平均位置
- その 2 つを結ぶ方向

ただし、manifold steering の論文が強調するように、概念の変化はいつも一直線とは限りません。政策コストの議論、科学的不確実性、科学者不信、明確な否認論は、自然な「道」の上で少しずつ移り変わる可能性があります。


In [ ]:
#@title 図 5: 表現空間: 否認論方向と出力確率を重ねる {display-mode: "form"}
labels = claims_df["denial_label"].to_numpy().astype(bool)
consensus_mean = claim_matrix[~labels].mean(axis=0)
denial_mean = claim_matrix[labels].mean(axis=0)
denial_axis = denial_mean - consensus_mean
denial_axis = denial_axis / (np.linalg.norm(denial_axis) + 1e-8)
global_claim_center = claim_matrix.mean(axis=0)
axis_scores = (claim_matrix - global_claim_center) @ denial_axis
# Normalize only for display.
axis_scores01 = (axis_scores - axis_scores.min()) / (axis_scores.max() - axis_scores.min() + 1e-9)
claims_df["activation_denial_axis"] = axis_scores01

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.9))

# Left: internal axis score by claim group.
order = claims_df.sort_values("activation_denial_axis").reset_index(drop=True)
y = np.arange(len(order))
point_colors = [OI["orange"] if v == 1 else OI["blue"] for v in order["denial_label"]]
axes[0].hlines(y, 0, order["activation_denial_axis"], color="#CBD0D8", linewidth=3)
axes[0].scatter(order["activation_denial_axis"], y, c=point_colors, s=70, edgecolor="white", linewidth=1.0, zorder=3)
axes[0].set_yticks(y)
axes[0].set_yticklabels([short_label(g + ": " + claim, 44) for g, claim in zip(order["group"], order["claim"])], fontsize=8.2)
axes[0].set_xlim(0, 1.02)
axes[0].set_xlabel("内部表現の denial axis score")
axes[0].set_title("内部表現にも、合意/否認を分ける方向が見える")
clean_ax(axes[0], grid="x")

# Right: relationship between internal axis and behavior probability.
axes[1].scatter(
    claims_df["activation_denial_axis"], claims_df["qwen_p_denialism"],
    c=[OI["orange"] if v == 1 else OI["blue"] for v in claims_df["denial_label"]],
    s=95, edgecolor="white", linewidth=1.2, zorder=3,
)
for _, r in claims_df.iterrows():
    if r["group"] in ["science skepticism", "science unreliable", "policy concern"]:
        axes[1].text(r["activation_denial_axis"] + 0.015, r["qwen_p_denialism"] + 0.02, short_label(r["group"], 18), fontsize=8, color="#444444")
axes[1].axhline(0.5, color="#333333", linestyle="--", linewidth=1)
axes[1].set_xlim(-0.04, 1.04)
axes[1].set_ylim(-0.04, 1.04)
axes[1].set_xlabel("内部表現の denial axis score")
axes[1].set_ylabel("Qwen P(denialism = 1)")
axes[1].set_title("表現空間の方向と、出力確率を対応づける")
clean_ax(axes[1])

legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=OI["blue"], markeredgecolor="white", markersize=9, label="非否認 / 合意・政策懸念"),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=OI["orange"], markeredgecolor="white", markersize=9, label="否認論"),
]
axes[1].legend(handles=legend_handles, loc="lower right")
add_note(fig, "denial axis は、この小さな主張バンクの平均差分。研究では別データで検証する必要がある。")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


In [ ]:
#@title 図 6: 一直線ではなく「境界をまたぐ道」として見る {display-mode: "form"}
path_df = claims_df.sort_values("stance").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.2, 6.0))
ax.plot(path_df["activation_denial_axis"], path_df["qwen_p_denialism"], color="#222222", linewidth=2.0, alpha=0.85, zorder=2)
sc = ax.scatter(
    path_df["activation_denial_axis"], path_df["qwen_p_denialism"],
    c=path_df["stance"], cmap="cividis", s=110, edgecolor="white", linewidth=1.2, zorder=3,
)

# Straight line between endpoints: the simple steering intuition.
start = path_df.iloc[0]
end = path_df.iloc[-1]
ax.plot(
    [start["activation_denial_axis"], end["activation_denial_axis"]],
    [start["qwen_p_denialism"], end["qwen_p_denialism"]],
    linestyle="--", color=OI["gray"], linewidth=2.0, label="単純な直線"
)
ax.axhline(0.5, color="#333333", linestyle=":", linewidth=1.2)
ax.text(0.04, 0.54, "分類境界", fontsize=9, color="#333333")
ax.set_title("manifold steering の直感: 概念変化は、自然な道に沿って急に曲がることがある")
ax.set_xlabel("内部表現の denial axis score")
ax.set_ylabel("行動空間: P(denialism = 1)")
ax.set_xlim(-0.04, 1.04)
ax.set_ylim(-0.04, 1.04)
clean_ax(ax)
ax.legend(loc="lower right")
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label("stance ladder")
add_note(fig, "黒線は観察された主張の順序。灰色の破線は始点と終点だけを結ぶ単純な方向。")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## セクション 3 のまとめ

このセクションで見たのは、研究の完成形ではなく、メカニズム分析の入口です。

それでも、メディア研究にとって重要な示唆があります。

- 1 回の出力ではなく、`P(denialism=1)` を見ると、モデルがどこで境界を引いているかが見える
- Qwen3-8B の内部表現にも、合意主張と否認主張を分ける方向が見える
- 「政策コストへの懸念」と「科学者不信」は、見た目は似ていても、モデルの判定では違う場所に置かれることがある
- manifold steering の発想は、「正しい方向を 1 本探す」だけでなく、「自然な概念変化の道」を探すことへ研究者の目を向ける

研究として進めるなら、主張数を増やし、複数レイヤーを比較し、別データで検証し、複数モデルで同じ構造が出るかを確かめます。


---

# 今日のまとめ

今日のノートブックでは、LLM をメディア研究に応用する 3 つの入口を見ました。

1. **代理ラベリング**: 大量テキストの内容分析を助ける。ただし 1 回のラベルではなく、確率、不確実性、人間ラベルとの較正を見る
2. **社会シミュレーション**: ネットワーク、信念、役割、情報源を組み合わせ、専門家介入や誤情報の仮説を試す
3. **メカニズム分析**: LLM がどこで境界を引くのかを、出力確率と内部表現の両方から観察する

LLM は「研究者の代わりに結論を出す機械」ではありません。むしろ、問いを広げ、仮説を可視化し、検証すべき境界事例を見つけるための道具です。


## ミニ演習

時間があれば、次のどれかを試してください。

1. 代理ラベリングで、しきい値を `0.5` から `0.7` に変えると、偽陽性と偽陰性はどう変わるでしょうか。
2. 社会シミュレーションで、`homophily` を強くすると、分極化は速く進むでしょうか。
3. AI アシスタントの悪意ある投稿を、毎日 1 回ではなく 3 回入れると、脆弱性ギャップはどう変わるでしょうか。
4. メカニズム分析で、気候主張を 20 個に増やすと、PCA の道は直線に近づくでしょうか、それとも曲がって見えるでしょうか。

演習のポイントは「きれいな答えを出すこと」ではなく、**モデルの結果がどの設定に敏感かを観察すること**です。


---

## 発展: もっと深く読みたい人へ

今日のノートブックは入口です。興味がある人は、次の順番で読むと理解しやすいです。

**代理ラベリング / 内容分析**

- 大事な問い: LLM のラベルを人間ラベルの代わりに使うとき、どのくらい較正・検証すべきか。
- このノートブックで見たこと: 1 回の返答ではなく、次トークン確率や複数回サンプリングを見る。
- 発展課題: 気候否認論以外のコードブックを作り、境界事例だけを人間が再確認する workflow を設計する。

**社会シミュレーション / FUSE**

- Liu, Y., Song, Z., Zhang, J., Zhang, X., Chen, X., & Yan, R. (2025). *The Stepwise Deception: Simulating the Evolution from True News to Fake News with LLM Agents.* arXiv:2410.19064. https://arxiv.org/abs/2410.19064
- 大事な問い: 事実そのものが最初から偽なのではなく、ネットワークの中で少しずつ歪む過程をどう観察するか。
- このノートブックで見たこと: 役割、信念、ネットワーク、専門家介入、誤情報投稿を小さなモデルとして組み合わせる。
- 発展課題: 専門家を 1 人ではなく複数置く、橋渡し tie の数を変える、検証者の割合を増やす。

**メカニズム分析 / Manifold steering**

- Wurgaft, D., Rager, C., Kowal, M., Shyam, V., Feucht, S., Bhalla, U., Haklay, T., Bigelow, E., Sarfati, R., McGrath, T., Lewis, O., Merullo, J., Goodman, N., Fel, T., Geiger, A., & Lubana, E. S. (2026). *Manifold Steering Reveals the Shared Geometry of Neural Network Representation and Behavior.* arXiv:2605.05115. https://arxiv.org/abs/2605.05115
- 大事な問い: モデル内部の表現空間で、概念は 1 本の直線として動くのか、それとも曲がった道として動くのか。
- このノートブックで見たこと: 気候主張を合意から否認論へ並べると、出力確率と内部表現の両方に境界が見える。
- 発展課題: claim bank を 100 件以上に増やし、denial axis の安定性、境界の位置、曲率、モデル間差を調べる。

**研究者向けの注意**

このノートブックの結果は「発見の入口」です。研究として主張するには、サンプルを増やし、別のコードブックで再現し、複数のランダムシード、複数モデル、複数プロンプトで頑健性を確かめる必要があります。LLM は便利ですが、研究者の判断を置き換えるものではありません。
